# 07 - Genie space over silver + gold tables (step 8)

Creates or updates the Genie space **FE Bar Fraud RT - Silver Events** over the four silver tables (`silver_transfers`, `silver_login`, `silver_otp`, `silver_screen_view`) and the two gold tables built by `08_gold_tables` (`gold_scored_transfers`, `gold_daily_fraud_summary`).

* **Scope:** silver events plus model scores. The space started as silver only; gold was added later so analysts can ask about scores, flags and flagged-fraud capture. The title is kept so the space id and its links stay stable. Analyst decisions stay in the Lakebase review queue (served by the app), not in Genie.
* **Curation:** technical columns (`sequence_number`, `partition_key`, `_ingested_at`, `_source_file`) are hidden; PII columns are described as masked; categorical columns use entity matching.
* **Instructions:** fraud-rate definition, failed login/OTP definitions, join key, and "relative windows are measured from the table's latest event" because the data is a synthetic backfill plus occasional live bursts.
* **Example SQL:** 7 question/SQL pairs (5 silver, 2 gold), each executed below before the space is written so Genie never learns a broken query.
* **Idempotent:** finds the space by title and replaces its config (full replacement), otherwise creates it.

In [0]:
import json, time
from databricks.sdk import WorkspaceClient

dbutils.widgets.text("catalog", "mlo_serverless_stable_catalog")
dbutils.widgets.text("schema", "fraud_rt")
dbutils.widgets.text("warehouse_id", "bf8a061e0b800955")  # Serverless Starter Warehouse
CATALOG, SCHEMA = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
WAREHOUSE_ID = dbutils.widgets.get("warehouse_id")
C = f"{CATALOG}.{SCHEMA}"
TITLE = "FE Bar Fraud RT - Silver Events"
PARENT = "/Workspace/Users/mike.lo@databricks.com/rt-fraud-fs"
w = WorkspaceClient()
print(C, WAREHOUSE_ID)

In [0]:
EXAMPLES = [
 ("What is the fraud rate by corridor?",
f"""SELECT corridor, COUNT(*) AS transfers, SUM(is_fraud) AS fraud_transfers,
       ROUND(AVG(is_fraud) * 100, 2) AS fraud_rate_pct,
       ROUND(SUM(CASE WHEN is_fraud = 1 THEN amount_usd ELSE 0 END), 2) AS fraud_amount_usd
FROM {C}.silver_transfers
GROUP BY corridor
ORDER BY fraud_rate_pct DESC"""),
 ("Which customers had the most failed OTPs in the last 24 hours?",
f"""SELECT customer_id, COUNT(*) AS failed_otps, MAX(event_ts) AS last_failed_otp_ts
FROM {C}.silver_otp
WHERE status = 'failed'
  AND event_ts >= (SELECT MAX(event_ts) FROM {C}.silver_otp) - INTERVAL 24 HOURS
GROUP BY customer_id
ORDER BY failed_otps DESC
LIMIT 20"""),
 ("Compare average transfer amount for fraud vs legitimate transfers by payout method",
f"""SELECT payout_method,
       CASE WHEN is_fraud = 1 THEN 'fraud' ELSE 'legitimate' END AS label,
       COUNT(*) AS transfers, ROUND(AVG(amount_usd), 2) AS avg_amount_usd
FROM {C}.silver_transfers
GROUP BY payout_method, label
ORDER BY payout_method, label"""),
 ("Do fraud transfers follow failed OTP attempts in the hour before the transfer?",
f"""WITH t AS (
  SELECT transfer_id, customer_id, ts, is_fraud FROM {C}.silver_transfers
), o AS (
  SELECT t.transfer_id, COUNT(o.event_id) AS failed_otps_1h_before
  FROM t LEFT JOIN {C}.silver_otp o
    ON o.customer_id = t.customer_id AND o.status = 'failed'
   AND o.event_ts >= t.ts - INTERVAL 1 HOUR AND o.event_ts < t.ts
  GROUP BY t.transfer_id
)
SELECT CASE WHEN t.is_fraud = 1 THEN 'fraud' ELSE 'legitimate' END AS label,
       COUNT(*) AS transfers,
       ROUND(AVG(o.failed_otps_1h_before), 3) AS avg_failed_otps_1h_before,
       ROUND(AVG(CASE WHEN o.failed_otps_1h_before > 0 THEN 1 ELSE 0 END) * 100, 2) AS pct_with_failed_otp
FROM t JOIN o USING (transfer_id)
GROUP BY label"""),
 ("How many fraud transfers did the model flag, and how many were missed?",
f"""SELECT SUM(CASE WHEN is_fraud = 1 AND flagged = 1 THEN 1 ELSE 0 END) AS fraud_flagged,
       SUM(CASE WHEN is_fraud = 1 AND flagged = 0 THEN 1 ELSE 0 END) AS fraud_missed,
       SUM(CASE WHEN is_fraud = 0 AND flagged = 1 THEN 1 ELSE 0 END) AS legit_flagged,
       ROUND(SUM(CASE WHEN is_fraud = 1 AND flagged = 1 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(flagged), 0), 2) AS flag_precision_pct,
       ROUND(SUM(CASE WHEN is_fraud = 1 AND flagged = 1 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(is_fraud), 0), 2) AS flag_recall_pct
FROM {C}.gold_scored_transfers"""),
 ("Which corridors had the most flagged transfers in the last 7 days?",
f"""SELECT corridor, SUM(flagged_transfers) AS flagged_transfers, SUM(flagged_fraud_transfers) AS flagged_fraud,
       SUM(transfers) AS transfers
FROM {C}.gold_daily_fraud_summary
WHERE day >= (SELECT MAX(day) FROM {C}.gold_daily_fraud_summary) - INTERVAL 7 DAYS
GROUP BY corridor
ORDER BY flagged_transfers DESC"""),
 ("Daily transfer volume and fraud count over time",
f"""SELECT DATE(ts) AS day, COUNT(*) AS transfers, SUM(is_fraud) AS fraud_transfers,
       ROUND(SUM(amount_usd), 2) AS total_amount_usd
FROM {C}.silver_transfers
GROUP BY day
ORDER BY day"""),
]

# Run every example before it is written to the space.
for q, s in EXAMPLES:
    pdf = spark.sql(s).toPandas()
    assert len(pdf) > 0, f"empty result for: {q}"
    print(f"== {q}  ({len(pdf)} rows)")
    print(pdf.head(5).to_string(index=False))

In [0]:
# Deterministic ids so re-runs replace the same items instead of creating new ones.
_n = iter(range(1, 1000))
def nid(): return f"01f0c0de{next(_n):024x}"

HIDE = ["sequence_number", "partition_key", "_ingested_at", "_source_file"]
def cols(extra):
    return sorted([{"column_name": c, "exclude": True} for c in HIDE] + extra, key=lambda x: x["column_name"])
PII = lambda c, what: {"column_name": c, "description": [f"{what} (PII, masked for non-admins)"]}

tables = [
 {"identifier": f"{C}.silver_login",
  "description": ["One row per login attempt (synthetic). success=false is a failed login; failure_reason is 'bad_password' or null."],
  "column_configs": cols([{"column_name": "event_ts", "description": ["Login event time (UTC)"]},
                          {"column_name": "success", "description": ["true = successful login, false = failed login"]},
                          {"column_name": "geo_country", "enable_entity_matching": True},
                          {"column_name": "auth_method", "enable_entity_matching": True},
                          PII("device_id", "Device identifier"), PII("ip", "IP address")])},
 {"identifier": f"{C}.silver_otp",
  "description": ["One row per OTP event (synthetic). status is 'sent', 'failed' or 'verified'. Bursts of failed OTPs are an account-takeover signal."],
  "column_configs": cols([{"column_name": "event_ts", "description": ["OTP event time (UTC)"]},
                          {"column_name": "status", "description": ["sent, failed or verified"], "enable_entity_matching": True},
                          {"column_name": "otp_channel", "enable_entity_matching": True},
                          PII("phone", "Phone number"), PII("device_id", "Device identifier")])},
 {"identifier": f"{C}.silver_screen_view",
  "description": ["One row per app screen view (synthetic) with dwell time in ms. Rapid hops through security_settings, change_phone and add_recipient are a fraud signal."],
  "column_configs": cols([{"column_name": "event_ts", "description": ["Screen view time (UTC)"]},
                          {"column_name": "screen_name", "enable_entity_matching": True},
                          {"column_name": "dwell_ms", "description": ["Time spent on the screen in milliseconds"]},
                          PII("device_id", "Device identifier")])},
 {"identifier": f"{C}.silver_transfers",
  "description": ["One row per cross-border remittance transfer (synthetic). is_fraud is the synthetic ground-truth label (1 = fraud)."],
  "column_configs": cols([{"column_name": "ts", "description": ["Transfer time (UTC)"], "synonyms": ["transfer time", "transfer date"]},
                          {"column_name": "amount_usd", "description": ["Transfer amount in USD"], "synonyms": ["amount", "value"]},
                          {"column_name": "corridor", "description": ["Send-receive country pair, e.g. US-MX"], "enable_entity_matching": True},
                          {"column_name": "payout_method", "enable_entity_matching": True},
                          {"column_name": "is_fraud", "description": ["1 = fraud, 0 = legitimate (synthetic label)"], "synonyms": ["fraud flag", "label"]},
                          PII("device_id", "Device identifier"), PII("ip", "IP address")])},
 {"identifier": f"{C}.gold_scored_transfers",
  "description": ["One row per transfer scored by transfer_fraud_model with point-in-time features. fraud_score is the model probability; flagged = 1 when the transfer is in the top-scored share that enters the analyst review queue. Scores are in-sample for history (synthetic)."],
  "column_configs": sorted([
      {"column_name": "sequence_number", "description": ["Unique key; transfer_id can repeat across live runs"]},
      {"column_name": "fraud_score", "description": ["Model fraud probability 0-1"], "synonyms": ["score", "risk score", "model score"]},
      {"column_name": "flagged", "description": ["1 = flagged for analyst review (fraud_score > score_threshold, top 2% of scores)"], "synonyms": ["alert", "flag"]},
      {"column_name": "is_fraud", "description": ["1 = fraud, 0 = legitimate (synthetic label)"]},
      {"column_name": "corridor", "enable_entity_matching": True},
      {"column_name": "payout_method", "enable_entity_matching": True},
      {"column_name": "ts", "description": ["Transfer time (UTC)"]},
      {"column_name": "scored_at", "exclude": True}], key=lambda x: x["column_name"])},
 {"identifier": f"{C}.gold_daily_fraud_summary",
  "description": ["One row per day, corridor and payout method: transfers, fraud, flagged and flagged-fraud counts and USD amounts (from gold_scored_transfers). Prefer this table for daily trends."],
  "column_configs": sorted([
      {"column_name": "day", "description": ["Transfer date (UTC)"]},
      {"column_name": "corridor", "enable_entity_matching": True},
      {"column_name": "payout_method", "enable_entity_matching": True},
      {"column_name": "flagged_fraud_transfers", "description": ["Fraud transfers that the model flagged (caught)"]}], key=lambda x: x["column_name"])},
]

TEXT = f"""This space covers SYNTHETIC data for a cross-border remittance fraud prototype (FE Bar). All tables are in {C}.
- Join tables on customer_id. Transfers use ts for time; login, otp and screen_view use event_ts. All timestamps are UTC.
- Fraud rate = AVG(is_fraud) on silver_transfers, shown as a percentage rounded to 2 decimals. Fraud amount = SUM(amount_usd) where is_fraud = 1.
- A failed login is silver_login.success = false. A failed OTP is silver_otp.status = 'failed'.
- Data is a 30-day backfill plus occasional live bursts, so relative windows ("last 24 hours", "last hour") should be measured from MAX(event_ts) or MAX(ts) of the table, not from current_timestamp(), unless the user asks for wall-clock time.
- device_id, ip and phone are PII and are masked for non-admin users; count distinct values rather than listing them.
- Corridors: US-MX, US-PH, US-IN, US-GT, US-NG. Payout methods: bank_deposit, cash_pickup, mobile_wallet, home_delivery.
- Round USD amounts to 2 decimals.
- Model scores: use gold_scored_transfers (fraud_score, flagged). Flag precision = flagged fraud / flagged; flag recall = flagged fraud / all fraud. Scores for past transfers are in-sample (the model was refit on all data); say so when reporting model performance.
- For daily or corridor trends prefer gold_daily_fraud_summary; for per-transfer or per-customer questions use gold_scored_transfers or silver tables."""

SAMPLE_QUESTIONS = [
    "What is the fraud rate by corridor?",
    "Which customers had the most failed OTPs in the last 24 hours?",
    "Show daily transfer volume and fraud count",
    "Compare average transfer amount for fraud vs legitimate transfers by payout method",
    "Which screens do fraudsters visit most compared to legitimate customers?",
    "How many fraud transfers did the model flag, and how many were missed?",
    "Which corridors had the most flagged transfers in the last 7 days?",
]

space = {
 "version": 2,
 "config": {"sample_questions": sorted([{"id": nid(), "question": [q]} for q in SAMPLE_QUESTIONS], key=lambda x: x["id"])},
 "data_sources": {"tables": sorted(tables, key=lambda t: t["identifier"])},
 "instructions": {
   "text_instructions": [{"id": nid(), "content": [TEXT]}],
   "example_question_sqls": sorted([{"id": nid(), "question": [q], "sql": [s]} for q, s in EXAMPLES], key=lambda x: x["id"]),
   "join_specs": sorted([{"id": nid(),
       "left": {"identifier": f"{C}.silver_transfers", "alias": "silver_transfers"},
       "right": {"identifier": f"{C}.{t}", "alias": t},
       "sql": [f"`silver_transfers`.`customer_id` = `{t}`.`customer_id`", "--rt=FROM_RELATIONSHIP_TYPE_MANY_TO_MANY--"]}
       for t in ["silver_login", "silver_otp", "silver_screen_view"]], key=lambda x: x["id"]),
 },
}
print(json.dumps({k: (len(v) if isinstance(v, list) else {kk: len(vv) for kk, vv in v.items()}) for k, v in
                  {"tables": tables, "instructions": space["instructions"], "sample_questions": space["config"]["sample_questions"]}.items()}))

In [0]:
body = {"warehouse_id": WAREHOUSE_ID, "title": TITLE,
        "description": "Ask questions about synthetic remittance transfers, logins, OTPs and screen views (silver) and model fraud scores and review flags (gold) for the real-time fraud prototype.",
        "serialized_space": json.dumps(space)}

existing = [s for s in w.api_client.do("GET", "/api/2.0/genie/spaces").get("spaces", []) if s["title"] == TITLE]
if existing:
    SPACE_ID = existing[0]["space_id"]
    w.api_client.do("PATCH", f"/api/2.0/genie/spaces/{SPACE_ID}", body=body)
    print("updated", SPACE_ID)
else:
    SPACE_ID = w.api_client.do("POST", "/api/2.0/genie/spaces", body={**body, "parent_path": PARENT})["space_id"]
    print("created", SPACE_ID)

# Read back and compare with what was sent.
g = w.api_client.do("GET", f"/api/2.0/genie/spaces/{SPACE_ID}", query={"include_serialized_space": "true"})
s = json.loads(g["serialized_space"])
print("tables:", [t["identifier"].split(".")[-1] for t in s["data_sources"]["tables"]])
print("sample questions:", len(s["config"]["sample_questions"]), "| example SQLs:", len(s["instructions"]["example_question_sqls"]),
      "| join specs:", len(s["instructions"].get("join_specs", [])), "| text instructions:", len(s["instructions"]["text_instructions"]))

In [0]:
QUESTION = "What is the fraud rate by payout method?"
r = w.api_client.do("POST", f"/api/2.0/genie/spaces/{SPACE_ID}/start-conversation", body={"content": QUESTION})
cid, mid = r["conversation_id"], r["message_id"]
for _ in range(60):
    m = w.api_client.do("GET", f"/api/2.0/genie/spaces/{SPACE_ID}/conversations/{cid}/messages/{mid}")
    if m["status"] in ("COMPLETED", "FAILED", "CANCELLED"):
        break
    time.sleep(5)
print("status:", m["status"])

genie_rows = None
for a in m.get("attachments", []):
    if "query" in a:
        print(a["query"]["query"])
        res = w.api_client.do("GET", f"/api/2.0/genie/spaces/{SPACE_ID}/conversations/{cid}/messages/{mid}/attachments/{a['attachment_id']}/query-result")
        cols_ = [c["name"] for c in res["statement_response"]["manifest"]["schema"]["columns"]]
        if "fraud_transfers" in cols_:
            genie_rows = {row[cols_.index("payout_method")]: int(row[cols_.index("fraud_transfers")])
                          for row in res["statement_response"]["result"]["data_array"]}
    if "text" in a:
        print(a["text"]["content"])

# Independent check: fraud transfers per payout method straight from the table.
truth = {r["payout_method"]: int(r["n"]) for r in spark.sql(
    f"SELECT payout_method, SUM(is_fraud) AS n FROM {C}.silver_transfers GROUP BY payout_method").collect()}
print("genie:", genie_rows)
print("sql:  ", truth)
print("MATCH" if genie_rows == truth else "CHECK MANUALLY (Genie used different columns or numbers)")